In [33]:
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
from meteostat import Point, Hourly, Stations
from tqdm import tqdm
import os
import sys
import numpy as np
from typing import Optional, Callable, List
from abc import ABC, abstractmethod

In [50]:
os.getcwd()
#list
print(os.listdir(os.getcwd()))

['Singular.ipynb', 'HourlyFetch.ipynb', '.DS_Store', 'RunMLflow.sh', 'SingleVanillaTransformer.ipynb', 'StationDensity.ipynb', 'CartographyBasic.ipynb', 'RunTraining.sh', 'Utils', '__pycache__', 'Conceputalise.pptx', 'REFLECTION.ipynb', 'mlruns', 'logs', 'FitExample.ipynb', 'git_include.txt', 'Train.py', 'VanillaTransformer.py', 'Abscissa.ipynb', 'BackToAlive.ipynb', 'DataPipelineWorkShop.py', '.vscode', 'train.log', '.empty']


In [34]:
sys.path.append('Utils/')
from PlotUtils import setMplParam, getColour, getHistoParam, annotate_histogram
from DataPipelineWorkShop import get_hourly_example, MeteoPreprocessor
# getHistoParam: 
# Nbins, binwidth, bins, counts, bin_centers  = 
# from ExternalFunctions import nice_string_output, add_text_to_ax
setMplParam()

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
# import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule, LightningModule, Trainer
# 30 sec
# 

In [ ]:
class MeteoDataset(Dataset):
    """
    Dataset for meteorological sequence-to-sequence forecasting.
    Robust to missing or all-NaN feature columns.

    Each sample consists of:
        - X: input window of shape (window_size, num_features)
        - Y: target window of shape (horizon, num_features)
    """

    def __init__(self,
                 source_data: pd.DataFrame,
                 feature_cols: List[str],
                 window_size: int = 24,
                 horizon: int = 6,
                 overlap: bool = True,
                 strict: bool = False):  # relaxed default
        self.feature_cols = feature_cols
        self.window_size = window_size
        self.horizon = horizon
        self.overlap = overlap
        self.strict = strict

        # --- Defensive copy ---
        self.source_data = source_data.copy()

        # --- Drop obvious metadata columns ---
        for col in ['station', 'time']:
            if col in self.source_data.columns:
                self.source_data = self.source_data.drop(columns=[col])

        # --- Convert nullable pandas dtypes → native float ---
        self.source_data = self.source_data.convert_dtypes()
        for col in self.source_data.columns:
            if pd.api.types.is_extension_array_dtype(self.source_data[col].dtype):
                self.source_data[col] = self.source_data[col].astype(float)

        # --- Identify usable columns ---
        self.available_features = self._validate_features()

        # --- Subset to usable columns ---
        self.source_data = self.source_data[self.available_features]

        # --- Convert to NumPy float32 array ---
        self.values = self.source_data.values.astype(np.float32)

        # --- Build sliding windows ---
        self.windows = self._make_windows()

        # === 🧾 Print summary info ===
        if len(self.windows) > 0:
            x0, y0 = self.windows[0]
            print(f"📊 MeteoDataset built: {len(self.windows)} samples | "
                  f"Input shape: {x0.shape} | Target shape: {y0.shape} | "
                  f"Features: {len(self.available_features)} ({self.available_features})")
        else:
            print("⚠️ MeteoDataset built with 0 samples!")

    # ==============================================================
    def _validate_features(self) -> List[str]:
        """Check available columns and drop missing / all-NaN ones."""
        missing = [c for c in self.feature_cols if c not in self.source_data.columns]
        all_nan = [c for c in self.feature_cols
                   if c in self.source_data.columns and self.source_data[c].isna().all()]

        available = [c for c in self.feature_cols
                     if c in self.source_data.columns and c not in all_nan]

        if missing:
            print(f"⚠️ Missing columns ignored: {missing}")
        if all_nan:
            print(f"⚠️ Columns all NaN dropped: {all_nan}")
        if not available:
            raise ValueError("No valid feature columns remain after filtering.")

        return available

    # ==============================================================
    def _make_windows(self):
        """Construct (X, Y) window pairs."""
        X, Y = [], []
        step = 1 if self.overlap else self.window_size
        n = len(self.values)
        for i in range(0, n - self.window_size - self.horizon + 1, step):
            x = self.values[i : i + self.window_size]
            y = self.values[i + self.window_size : i + self.window_size + self.horizon]
            X.append(x)
            Y.append(y)
        return list(zip(X, Y))

    # ==============================================================
    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        x, y = self.windows[idx]
        x_tensor = torch.tensor(x, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32)

        # Create masks: True where data is valid
        x_mask = ~torch.isnan(x_tensor)
        y_mask = ~torch.isnan(y_tensor)

        # Optionally replace NaNs with zeros for numerical stability
        x_tensor = torch.nan_to_num(x_tensor, nan=0.0)
        y_tensor = torch.nan_to_num(y_tensor, nan=0.0)

        return x_tensor, y_tensor, x_mask, y_mask

In [37]:
kbh = Point(lat=55.6761, lon=12.5683)
kbh_df = get_hourly_example(kbh, start=datetime(2015, 1, 1), end=datetime(2018, 12, 31))
# 2020.01.01 - 2021.12.31 : 14 sec
# 2015.01.01 - 2018.12.31 : 41 sec

In [38]:
processed_df = MeteoPreprocessor()(kbh_df)
feature_list = processed_df.columns.tolist()  # after transformation
processed_df.shape

(35041, 18)

In [47]:
processed_df.columns

Index(['temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wspd', 'wpgt', 'pres', 'tsun',
       'coco', 'sin_hour', 'cos_hour', 'sin_week', 'cos_week', 'sin_year',
       'cos_year', 'sin_wdir', 'cos_wdir'],
      dtype='object')

In [ ]:
['temp', 'dwpt', 'rhum', 'prcp', 'snow', 'wspd', 'wpgt', 'pres', 'tsun','coco', 'sin_hour', 'cos_hour', 'sin_week', 'cos_week', 'sin_year', 'cos_year', 'sin_wdir', 'cos_wdir']
['temp', 'dwpt', 'rhum', 'prcp', 'wspd', 'pres', 'coco',                        'sin_hour', 'cos_hour', 'sin_week', 'cos_week', 'sin_year', 'cos_year', 'sin_wdir', 'cos_wdir']

In [39]:
dataset = MeteoDataset(
    source_data=processed_df,
    feature_cols=feature_list,
    window_size=48,
    horizon=24,
    overlap=True
)

⚠️ Columns all NaN dropped: ['snow', 'tsun']


In [40]:
class MeteoDatasetModule(LightningDataModule):
    """
    Time-aware DataModule for Meteostat forecasting.
    Prevents leakage by inserting temporal gaps between train, val, and test.
    """

    def __init__(self,
                 data: pd.DataFrame,
                 feature_cols: List[str],
                 window_size: int = 48,
                 horizon: int = 24,
                 gap: int = 60,
                 batch_size: int = 32,
                 num_workers: int = 4,
                 val_ratio: float = 0.1,
                 test_ratio: float = 0.1,
                 shuffle_train: bool = True):
        super().__init__()

        self.data = data
        self.feature_cols = feature_cols
        self.window_size = window_size
        self.horizon = horizon
        self.gap = gap
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.val_ratio = val_ratio
        self.test_ratio = test_ratio
        self.shuffle_train = shuffle_train

        self.train_dataset = None
        self.val_dataset = None
        self.test_dataset = None

    # ===========================================================
    def setup(self, stage: Optional[str] = None):
        """
        Chronologically split with optional 'gap' zones between splits.
        Gaps ensure no window from training overlaps with validation/test data.
        """
        n = len(self.data)
        n_test = int(n * self.test_ratio)
        n_val = int(n * self.val_ratio)
        n_train = n - n_val - n_test

        # compute gap indices (avoid overlap)
        train_end = n_train - self.gap
        val_start = n_train + self.gap
        val_end = n_train + n_val - self.gap
        test_start = n_train + n_val + self.gap

        # enforce boundaries
        train_end = max(train_end, 0)
        val_start = min(val_start, n)
        val_end = min(val_end, n)
        test_start = min(test_start, n)

        # slice subsets
        df_train = self.data.iloc[:train_end]
        df_val = self.data.iloc[val_start:val_end]
        df_test = self.data.iloc[test_start:]

        # sanity check (no overlap)
        assert df_train.index.max() < df_val.index.min(), "Train–Val overlap detected!"
        assert df_val.index.max() < df_test.index.min(), "Val–Test overlap detected!"

        # instantiate datasets
        self.train_dataset = MeteoDataset(df_train, self.feature_cols, self.window_size, self.horizon)
        self.val_dataset = MeteoDataset(df_val, self.feature_cols, self.window_size, self.horizon)
        self.test_dataset = MeteoDataset(df_test, self.feature_cols, self.window_size, self.horizon)

    # ===========================================================
    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=self.shuffle_train,
            num_workers=self.num_workers,
            drop_last=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )


In [41]:
class MultiHeadAttention(nn.Module):
    """
    Multi-head self-attention with built-in NaN and causal masking support.
    """

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        # QKV linear projections
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        # Output projection
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    # ==============================================================
    def _make_attention_mask(self, x_mask: torch.Tensor, causal: bool = False) -> torch.Tensor:
        """
        Build combined attention mask from:
          - x_mask: per-timestep validity mask, shape (B, S)
          - causal: if True, disallow attending to future tokens

        Returns:
          attn_mask: (B, 1, S, S), dtype=bool
        """
        B, S = x_mask.shape

        # Base mask from data validity (True = keep, False = mask out)
        attn_mask = x_mask.unsqueeze(1).unsqueeze(2)  # (B, 1, 1, S)

        if causal:
            # Lower-triangular mask: allow attending to self and past only
            causal_mask = torch.tril(torch.ones(S, S, dtype=torch.bool, device=x_mask.device))
            attn_mask = attn_mask & causal_mask.unsqueeze(0).unsqueeze(0)

        return attn_mask

    # ==============================================================
    def forward(self, x: torch.Tensor, mask: torch.Tensor = None, causal: bool = False) -> torch.Tensor:
        """
        Args:
            x: (B, S, D)
            mask: optional per-timestep validity mask (B, S) where True = valid
            causal: bool, apply causal (lookahead) masking if True

        Returns:
            Tensor of shape (B, S, D)
        """
        B, S, D = x.shape
        H = self.n_heads
        Dh = self.head_dim

        # Project Q, K, V
        q = self.q_proj(x).view(B, S, H, Dh).transpose(1, 2)  # (B, H, S, Dh)
        k = self.k_proj(x).view(B, S, H, Dh).transpose(1, 2)
        v = self.v_proj(x).view(B, S, H, Dh).transpose(1, 2)

        # Build attention mask
        attn_mask = None
        if mask is not None:
            attn_mask = self._make_attention_mask(mask, causal=causal)
            # Convert bool mask → float mask for scaled_dot_product_attention
            attn_mask = attn_mask.logical_not()  # True where to mask
            attn_mask = attn_mask.float().masked_fill(attn_mask, float("-inf"))

        # Core attention
        attn_output = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout.p if self.training else 0.0,
            is_causal=False  # handled manually above
        )

        # Combine heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, S, D)
        attn_output = self.out_proj(attn_output)
        attn_output = self.dropout(attn_output)

        return attn_output


In [42]:
class FFN(nn.Module):
    """
    Position-wise Feed-Forward Network (FFN) used inside Transformer encoder blocks.
    Expands and projects the feature dimension with non-linearity and dropout.

    Formula:
        FFN(x) = Dropout(W2 * Activation(W1 * x)) + residual

    Args:
        d_model (int): Input and output feature dimension.
        d_ff (int): Hidden expansion dimension (usually 2–4× d_model).
        dropout (float): Dropout probability.
        activation (str): Activation function: 'gelu', 'relu', or 'silu'.
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1, activation: str = 'gelu'):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

        if activation == 'gelu':
            self.activation = nn.GELU()
        elif activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'silu':
            self.activation = nn.SiLU()
        else:
            raise ValueError(f"Unsupported activation: {activation}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Tensor of shape (B, S, D_model)

        Returns:
            Tensor of shape (B, S, D_model)
        """
        x = self.fc1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x


In [43]:
class EncoderBlock(nn.Module):
    """
    Single Transformer encoder block:
    LayerNorm → MultiHeadAttention (+ residual)
    LayerNorm → Feed-Forward Network (+ residual)

    Supports both NaN masking and causal masking.

    Args:
        d_model (int): Input/hidden feature dimension.
        n_heads (int): Number of attention heads.
        d_ff (int): Feed-forward expansion dimension.
        dropout (float): Dropout probability.
        activation (str): Activation for FFN ('gelu', 'relu', or 'silu').
    """

    def __init__(self,
                 d_model: int,
                 n_heads: int,
                 d_ff: int,
                 dropout: float = 0.1,
                 activation: str = 'gelu'):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout=dropout)
        self.dropout1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FFN(d_model, d_ff, dropout=dropout, activation=activation)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self,
                x: torch.Tensor,
                mask: torch.Tensor = None,
                causal: bool = False) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (B, S, D)
            mask: Optional mask of shape (B, S) where True = valid
            causal: Whether to apply causal masking (for autoregressive forecasting)

        Returns:
            Tensor of shape (B, S, D)
        """

        # --- Multi-head attention sublayer ---
        residual = x
        x = self.norm1(x)
        attn_out = self.attn(x, mask=mask, causal=causal)
        x = residual + self.dropout1(attn_out)

        # --- Feed-forward sublayer ---
        residual = x
        x = self.norm2(x)
        ffn_out = self.ffn(x)
        x = residual + self.dropout2(ffn_out)

        return x


In [44]:
class MeteoVanillaTransformerEncoder(LightningModule):
    """
    Vanilla Transformer Encoder for meteorological forecasting.

    Features:
      - Learnable absolute positional embeddings
      - NaN-aware attention masking
      - Optional causal masking (for autoregressive forecasts)
      - Fully self-contained (no external Encoder class)
    """

    def __init__(self,
                 feature_dim: int,
                 d_model: int = 128,
                 n_heads: int = 8,
                 d_ff: int = 512,
                 num_layers: int = 4,
                 dropout: float = 0.1,
                 lr: float = 1e-4,
                 horizon: int = 24,
                 causal: bool = False,
                 activation: str = 'gelu',
                 max_seq_len: int = 512):
        super().__init__()
        self.save_hyperparameters()

        self.feature_dim = feature_dim
        self.d_model = d_model
        self.horizon = horizon
        self.lr = lr
        self.causal = causal

        # --- Input projection ---
        self.input_proj = nn.Linear(feature_dim, d_model)

        # --- Learnable absolute positional embedding ---
        self.pos_embedding = nn.Parameter(torch.zeros(1, max_seq_len, d_model))
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

        # --- Stack of encoder blocks ---
        self.layers = nn.ModuleList([
            EncoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout,
                activation=activation,
            )
            for _ in range(num_layers)
        ])

        # --- Normalisation and output projection ---
        self.final_norm = nn.LayerNorm(d_model)
        self.output_head = nn.Linear(d_model, feature_dim)

        # --- Loss ---
        self.loss_fn = nn.MSELoss()

    # ==============================================================
    def forward(self, x, mask=None):
        """
        Args:
            x: (B, S, F)
            mask: Optional (B, S) boolean mask where True = valid (non-NaN)

        Returns:
            Predicted sequence (B, S, F)
        """
        B, S, _ = x.shape

        # Input projection
        x = self.input_proj(x)

        # Add positional encoding (truncate if seq shorter than max_seq_len)
        x = x + self.pos_embedding[:, :S, :]

        # Pass through encoder blocks
        for layer in self.layers:
            x = layer(x, mask=mask, causal=self.causal)

        # Final normalisation and output projection
        x = self.final_norm(x)
        out = self.output_head(x)

        return out

    # ==============================================================
    def _compute_loss(self, preds, targets, mask):
        """Compute MSE loss with masking applied."""
        valid = mask.any(-1)
        preds_future = preds[:, -self.horizon:, :]
        targets_future = targets[:, -self.horizon:, :]
        return self.loss_fn(preds_future[valid], targets_future[valid])

    # ==============================================================
    def training_step(self, batch, batch_idx):
        x, y, x_mask, y_mask = batch
        preds = self.forward(x, mask=x_mask.any(-1))
        loss = self._compute_loss(preds, y, y_mask)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y, x_mask, y_mask = batch
        preds = self.forward(x, mask=x_mask.any(-1))
        loss = self._compute_loss(preds, y, y_mask)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y, x_mask, y_mask = batch
        preds = self.forward(x, mask=x_mask.any(-1))
        loss = self._compute_loss(preds, y, y_mask)
        self.log("test_loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr)


In [46]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import MLFlowLogger

# ===========================================================
def lock_and_load(config):
    print("torch.cuda.is_available():", torch.cuda.is_available())
    available_devices = list(range(torch.cuda.device_count()))
    print(f"Available CUDA devices: {available_devices}")

    if torch.cuda.is_available() and len(config.get("gpu", [])) > 0:
        requested_gpus = config.get("gpu", [])
        selected_gpu = int(requested_gpus[0]) if requested_gpus else 0

        if selected_gpu in available_devices:
            torch.cuda.empty_cache()
            print("🔥 LOCK AND LOAD! GPU ENGAGED! 🔥")
            device = torch.device(f"cuda:{selected_gpu}")
            torch.cuda.set_device(selected_gpu)
            torch.set_float32_matmul_precision("highest")
            print(f"Using GPU: {selected_gpu} (cuda:{selected_gpu})")
        else:
            print(f"⚠️ Warning: GPU {selected_gpu} is not available. Using CPU instead.")
            device = torch.device("cpu")
    else:
        device = torch.device("cpu")
        print("CUDA not available. Using CPU.")

    print(f"Selected device: {device}")
    return device


# ===========================================================
def setup_logger(output_dir):
    os.makedirs(output_dir, exist_ok=True)
    log_dir = os.path.join(output_dir, datetime.now().strftime("%Y%m%d_%H%M%S"))
    os.makedirs(log_dir, exist_ok=True)
    print(f"📁 Logging to: {log_dir}")
    return log_dir


# ===========================================================
def parse_args():
    parser = argparse.ArgumentParser(description="Train Transformer on Meteostat data")
    parser.add_argument("--gpu", nargs="+", default=[], help="List of GPU IDs to use")
    parser.add_argument("--window_size", type=int, default=48)
    parser.add_argument("--horizon", type=int, default=24)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--num_layers", type=int, default=4)
    parser.add_argument("--n_heads", type=int, default=8)
    parser.add_argument("--d_model", type=int, default=128)
    parser.add_argument("--d_ff", type=int, default=512)
    parser.add_argument("--dropout", type=float, default=0.1)
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--val_ratio", type=float, default=0.1)
    parser.add_argument("--test_ratio", type=float, default=0.1)
    parser.add_argument("--data_dir", type=str, default="./data")
    parser.add_argument("--log_dir", type=str, default="./logs")
    parser.add_argument("--experiment", type=str, default="meteo_transformer")  # === MLFLOW ===
    args, _ = parser.parse_known_args()
    return vars(args)


# ===========================================================
def run():
    config = parse_args()
    device = lock_and_load(config)
    log_dir = setup_logger(config["log_dir"])

    # === MLFLOW LOGGER ===
    mlf_logger = MLFlowLogger(
        experiment_name=config["experiment"],
        tracking_uri="http://127.0.0.1:5000"  # or "file:./mlruns" if you don't run `mlflow ui`
    )
    mlf_logger.log_hyperparams(config)

    # === Example data ===
    print("🌍 Fetching Meteostat hourly data for Copenhagen...")
    kbh = Point(lat=55.6761, lon=12.5683)
    df_raw = get_hourly_example(kbh, start=datetime(2015, 1, 1), end=datetime(2018, 12, 31))

    print("🧹 Preprocessing...")
    processed_df = MeteoPreprocessor()(df_raw)
    feature_list = processed_df.columns.tolist()

    print("📦 Building DataModule...")
    dm = MeteoDatasetModule(
        data=processed_df,
        feature_cols=feature_list,
        window_size=config["window_size"],
        horizon=config["horizon"],
        batch_size=config["batch_size"],
        val_ratio=config["val_ratio"],
        test_ratio=config["test_ratio"]
    )

    print("⚙️ Building model...")
    model = MeteoVanillaTransformerEncoder(
        feature_dim=len(feature_list),
        d_model=config["d_model"],
        n_heads=config["n_heads"],
        d_ff=config["d_ff"],
        num_layers=config["num_layers"],
        dropout=config["dropout"],
        lr=config["lr"],
        horizon=config["horizon"]
    )

    checkpoint_callback = ModelCheckpoint(
        dirpath=log_dir,
        filename="epoch{epoch:02d}-val_loss{val_loss:.4f}",
        save_top_k=3,
        monitor="val_loss",
        mode="min"
    )

    early_stopping = EarlyStopping(monitor="val_loss", patience=5, mode="min")

    # === Trainer ===
    trainer = Trainer(
        max_epochs=config["epochs"],
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        devices=config["gpu"] if torch.cuda.is_available() and config["gpu"] else 1,
        callbacks=[checkpoint_callback, early_stopping],
        default_root_dir=log_dir,
        log_every_n_steps=20,
        deterministic=True,
        logger=mlf_logger  # <=== MLFLOW LOGGER CONNECTED
    )

    print("🚀 Starting training...")
    trainer.fit(model, datamodule=dm)

    print("✅ Training complete.")
    print(f"📂 Checkpoints saved in: {log_dir}")




In [ ]:
run()

torch.cuda.is_available(): False
Available CUDA devices: []
CUDA not available. Using CPU.
Selected device: cpu
📁 Logging to: ./logs/20251031_120506
🌍 Fetching Meteostat hourly data for Copenhagen...
🧹 Preprocessing...
📦 Building DataModule...
⚙️ Building model...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/yhjo/miniconda3/envs/tempestransformer/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


🚀 Starting training...
⚠️ Columns all NaN dropped: ['snow', 'wpgt', 'tsun']


/Users/yhjo/miniconda3/envs/tempestransformer/lib/python3.12/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:751: Checkpoint directory /Users/yhjo/Desktop/[2025] Be Resilient/Resilience/logs/20251031_120506 exists and is not empty.

  | Name         | Type       | Params | Mode 
----------------------------------------------------
0 | input_proj   | Linear     | 2.4 K  | train
1 | layers       | ModuleList | 793 K  | train
2 | final_norm   | LayerNorm  | 256    | train
3 | output_head  | Linear     | 2.3 K  | train
4 | loss_fn      | MSELoss    | 0      | train
  | other params | n/a        | 65.5 K | n/a  
----------------------------------------------------
863 K     Trainable params
0         Non-trainable params
863 K     Total params
3.455     Total estimated model params size (MB)
69        Modules in train mode
0         Modules in eval mode


⚠️ Columns all NaN dropped: ['snow', 'tsun']
⚠️ Columns all NaN dropped: ['snow', 'tsun']


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/yhjo/miniconda3/envs/tempestransformer/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:428: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<string>", line 1, in <module>
  File "/Users/yhjo/miniconda3/envs/tempestransformer/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Users/yhjo/miniconda3/envs/tempestransformer/lib/python3.12/multiproce

SystemExit: 1